[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syaikhipin/kdd26-memdiag/blob/main/notebooks/4_three_probe_diagnostics.ipynb)

Open this notebook in Google Colab for a rendered, runnable tutorial view: https://colab.research.google.com/github/syaikhipin/kdd26-memdiag/blob/main/notebooks/4_three_probe_diagnostics.ipynb


# 4. Three-probe memory diagnostics

This notebook demonstrates the proposal's three probes:

1. Retrieval relevance analysis
2. Context utilization analysis
3. Failure root-cause analysis

In [ ]:
import sys
from pathlib import Path

def _find_pkg_dir(start):
 cur = Path(start).resolve()
 for cand in [cur, *cur.parents]:
 for sub in ("source", "experiment"):
 if (cand / sub / "run.py").exists():
 return cand / sub
 return None

PROJECT_ROOT = Path.cwd()
EXPERIMENT_DIR = _find_pkg_dir(PROJECT_ROOT)
if EXPERIMENT_DIR is None:
 raise FileNotFoundError("Could not locate source/run.py (or source/run.py). Run this notebook from the repository root, or `pip install -e source` first.")
PROJECT_ROOT = EXPERIMENT_DIR.parent
RESULTS_DIR = PROJECT_ROOT / "results"
sys.path.insert(0, str(EXPERIMENT_DIR))
print("EXPERIMENT_DIR =", EXPERIMENT_DIR)
print("Experiment code exists:", (EXPERIMENT_DIR / "run.py").exists())


## OpenAI-compatible LLM-as-judge option

The proposal includes an LLM-as-judge component. These notebooks default to offline deterministic diagnostics for low-resource reproducibility, but the framework supports an OpenAI-compatible endpoint via:

- `--backend openai-compatible`
- `--base-url http://127.0.0.1:8317/api/provider/codex/v1`
- `--model gpt-5.5`
- `OPENAI_API_KEY` or `--api-key`

Do not hardcode API keys in notebooks. In Colab, set the key through environment variables or secrets.

In [ ]:
# Optional LLM-as-judge configuration. This cell does not call the API.
OPENAI_COMPATIBLE_CONFIG = {
 "backend": "openai-compatible",
 "base_url": "http://127.0.0.1:8317/api/provider/codex/v1",
 "model": "gpt-5.5",
 "api_key_env": "OPENAI_API_KEY",
 "api_key_present": bool(os.environ.get("OPENAI_API_KEY")),
}
print(OPENAI_COMPATIBLE_CONFIG)

## External semantic/testing evaluators

The source code now supports `--eval-backend offline|rhesis|semantica|all`.

- `offline` is deterministic and dependency-free for reproducible tutorial runs.
- `rhesis` is optional and reads `RHESIS_API_KEY` from the environment only.
- `semantica` is optional and uses semantic extraction/provenance signals when installed.

Do not paste real keys into notebooks.

In [ ]:
from evaluators import OfflineSemanticEvaluator

record = {
 "question": "Which prior evidence supports the answer?",
 "gold_answer": "The retrieved memory contains the supporting evidence.",
 "answer": "retrieved_evidence_answerable",
 "retrieved_texts": ["supporting evidence from the memory store"],
 "retrieval_recall": 1.0,
 "evidence_hit": True,
}
result = OfflineSemanticEvaluator().evaluate(record)
print(result.to_record_fields())

In [ ]:
from diagnostics import precision_recall

retrieved = ["a", "b", "c"]
relevant = ["b", "d"]
print(precision_recall(retrieved, relevant))

In [ ]:
from diagnostics import locomo_retrieval_diagnostics, locomo_utilization_category

class Entry:
 def __init__(self, dia_id):
 self.metadata = {"dia_id": dia_id}

retrieved = [{"entry": Entry("42"), "score": 0.9}, {"entry": Entry("99"), "score": 0.7}]
diag = locomo_retrieval_diagnostics(retrieved, ["42", "43"])
print(diag)
print(locomo_utilization_category(diag["evidence_hit"], "offline_evidence_heuristic"))

In [ ]:
latest = sorted(RESULTS_DIR.glob("run_*_real_metrics.json"))[-1]
metrics = json.loads(latest.read_text())
print("Using", latest)
for dataset, summary in metrics["real"]["datasets"].items():
 best = max(summary["by_strategy"].items(), key=lambda kv: kv[1]["evidence_hit_rate"])
 print(dataset, "best_strategy=", best[0], "hit=", best[1]["evidence_hit_rate"], "failures=", best[1]["failure_modes"])

Guidance: use this as Exercise 1. Ask participants to identify whether a low score is caused by retrieval miss, partial evidence, or unused memory.